In [1]:
from rag_system import RealEstateRAG
from pathlib import Path
from mlx_lm import load

import pandas as pd

import warnings
warnings.filterwarnings("ignore")

/Users/egor/VS_GIT_repositories/BYTE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/egor/VS_GIT_repositories/BYTE/src/RAG/rag_system.py:25: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
import os
os.environ["HF_API_KEY"] = "None"

In [3]:
rag = RealEstateRAG(
    chroma_db_path="src/RAG/chroma_db",
    embedding_model="cointegrated/rubert-tiny2",
    collection_name="nedvijimost"
)

 Loading embedding model: cointegrated/rubert-tiny2


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 3303.50it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
query = "5-комнатная квартира 100-300 м2 в центре, до 100 млн рублей Ленинградский проспект, Петроградская сторона, Петроградский район, Санкт-Петербург"
print(f"\n Searching: {query}")
results = rag.search(query, top_k=10)
print(f" Found {len(results)} similar apartments.\n")


 Searching: 5-комнатная квартира 100-300 м2 в центре, до 100 млн рублей Ленинградский проспект, Петроградская сторона, Петроградский район, Санкт-Петербург
 Found 10 similar apartments.



In [12]:
for i, apt in enumerate(results, 1):
    print(f"{i}. Offer ID: {apt['metadata']['offer_id']}")
    print(f"   Similarity distance: {apt['distance']:.4f}")
    print(f"   Price: ₽ {apt['metadata']['price']:,.0f}")
    print(f"   Address: {apt['metadata']['address']}")
    print(f"   URL: {apt['metadata']['url']}\n")

1. Offer ID: 6239246398701815144
   Similarity distance: 0.2348
   Price: ₽ 50,000,000
   Address: Москва, Ленинградский проспект, ,  , 13с1А
   URL: https://realty.yandex.ru/offer/6239246398701815144/

2. Offer ID: 3454465350825848841
   Similarity distance: 0.2350
   Price: ₽ 83,665,480
   Address: None
   URL: https://realty.yandex.ru/offer/3454465350825848841/

3. Offer ID: 6638757770343091678
   Similarity distance: 0.2350
   Price: ₽ 160,000,000
   Address: None
   URL: https://realty.yandex.ru/offer/6638757770343091678/

4. Offer ID: 655159294397259776
   Similarity distance: 0.2361
   Price: ₽ 17,000,000
   Address: Москва, Мирской переулок, ,  , 4
   URL: https://realty.yandex.ru/offer/655159294397259776/

5. Offer ID: 2624412511671839267
   Similarity distance: 0.2373
   Price: ₽ 72,000,000
   Address: None
   URL: https://realty.yandex.ru/offer/2624412511671839267/

6. Offer ID: 7733253580826961363
   Similarity distance: 0.2393
   Price: ₽ 85,221,200
   Address: None
   URL

In [13]:
# model_path = Path("./models/phi3-mini-8bit").resolve()

# # Verify the path exists
# print(f"Model path: {model_path}")
# print(f"Exists: {model_path.exists()}")

# if model_path.exists():
#     print(f"Contents: {list(model_path.glob('*'))[:5]}...")
# model, tokenizer = load(str(model_path))
# print("Model and tokenizer loaded successfully.")

In [14]:
if results:
    target = results[0]
    competitors = results[1:5]
    
    print("📊 Generating Deal Analysis Report...\n")
    report = rag.generate_report(
        target_apartment=results[0],
        similar_apartments=results[1:],
        use_llm=True,
        llm_backend="huggingface",
        api_key=os.environ["HF_API_KEY"],
        model_name="meta-llama/Llama-3.1-8B-Instruct" 
    )
    print(report)

📊 Generating Deal Analysis Report...

ПРОМТ 
 
 ЦЕЛЕВАЯ КВАРТИРА:
                • Адрес: Москва, Ленинградский проспект, ,  , 13с1А
                • Цена: 50,000,000 ₽
                • Площадь: 80.5 м²
                • Цена за м²: 621,118 ₽
                • Метро: Белорусская
                • Комнат: 3

                РЫНОК (аналоги):
                • Средняя цена: 79,442,927 ₽
                • Средняя цена за м²: 848,610 ₽
                • Отклонение цены: -37.1% 
 
1. ВЫГОДА: Да. Цена целевой квартиры на 37,1% ниже средней цены рынка.

2. ПЛЮСЫ: Цена за м² ниже средней цены рынка, метро Белорусская расположена в центре города, что делает доступность и транспортную доступность квартиры выше, чем у конкурентов.

3. МИНУСЫ: Площадь квартиры ниже средней площади конкурентов, что может быть недостатком для некоторых покупателей.

4. ВЕРДИКТ: ✅ Хорошая сделка

5. КОММЕНТАРИЙ: Для покупателя, который ищет недорогую и удобную квартиру в центре Москвы, целевая квартира может быть х